In [1]:
partition = 100

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
tested_configs = load_tested_configs(log_path)
#Running: n_tree=100, t_depth=9, hd=768, batch_size=256, feature_rate=0.4, dropout=0.0, lr=0.01

n_tree_values = [100]
tree_depth_values = [9]
hidden_dim = [768]
batch_size_values = [256]
tree_feature_rates = [0.4]
feat_dropouts = [0.0]
lrs = [0.01]

n_iter = 300
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))


available_configs = [cfg for cfg in param_space if cfg not in tested_configs]
sampled_configs = random.sample(param_space, min(n_iter, len(available_configs)))

best_acc = 0

sampled_configs = random.sample(param_space, min(n_iter, len(param_space)))
i = 1
for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in product(n_tree_values, tree_depth_values, hidden_dim, batch_size_values, tree_feature_rates, feat_dropouts, lrs):
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")

    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i =i + 1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(f"Best accuracy: {best_acc}")



Running: n_tree=100, t_depth=9, hd=768, batch_size=256, feature_rate=0.4, dropout=0.0, lr=0.01
1 / 100
Use gtd100 dataset
Patience: 100


Training Epochs:  53%|█████▎    | 211/400 [02:56<02:37,  1.20it/s]

Early stopping at epoch 212

Best Accuracy: 0.864286

Best hyperparameter configuration:
{'n_tree': 100, 'tree_depth': 9, 'batch_size': 256, 'hidden_dim': 768, 'tree_feature_rate': 0.4, 'feat_dropout': 0.0, 'lr': 0.01}
Best accuracy: 0.8642857142857143


In [5]:
#{'n_tree': 100, 'tree_depth': 9, 'batch_size': 512, 'hidden_dim': 768, 'tree_feature_rate': 0.4, 'feat_dropout': 0.0, 'lr': 0.01}


In [6]:
"""

========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd100
  Hidden Dim: 768
  n_tree: 100, tree_depth: 9, tree_feature_rate: 0.4
  Batch size: 128, Dropout: 0.0, LR: 0.01

Best Accuracy: 0.8489
Weighted Precision: 0.8585, Recall: 0.8489, F1 Score: 0.8464, ROCAUC: 0.9942
Macro Precision: 0.8585, Recall: 0.8489, F1 Score: 0.8464, ROCAUC: 0.9942
Micro Precision: 0.8489, Recall: 0.8489, F1 Score: 0.8489, ROCAUC: 0.9955
"""

'\n\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd100\n  Hidden Dim: 768\n  n_tree: 100, tree_depth: 9, tree_feature_rate: 0.4\n  Batch size: 128, Dropout: 0.0, LR: 0.01\n\nBest Accuracy: 0.8489\nWeighted Precision: 0.8585, Recall: 0.8489, F1 Score: 0.8464, ROCAUC: 0.9942\nMacro Precision: 0.8585, Recall: 0.8489, F1 Score: 0.8464, ROCAUC: 0.9942\nMicro Precision: 0.8489, Recall: 0.8489, F1 Score: 0.8489, ROCAUC: 0.9955\n'

In [7]:
#Running: n_tree=20, t_depth=9, hd=1024, batch_size=256, feature_rate=0.4, dropout=0.0, lr=0.01
#89.04

In [8]:
sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()

Use gtd100 dataset
Patience: 300


Training Epochs:   3%|▎         | 50/1500 [00:42<18:37,  1.30it/s]

[Epoch 50] Train Loss: 0.6455, Eval Loss: 0.7269, Eval Accuracy: 0.8381


Training Epochs:   7%|▋         | 100/1500 [01:20<17:41,  1.32it/s]

[Epoch 100] Train Loss: 0.4757, Eval Loss: 0.5812, Eval Accuracy: 0.8476


...

[Epoch 550] Train Loss: 0.3807, Eval Loss: 0.5060, Eval Accuracy: 0.8524


Training Epochs:  40%|████      | 600/1500 [07:43<11:41,  1.28it/s]

[Epoch 600] Train Loss: 0.3813, Eval Loss: 0.5046, Eval Accuracy: 0.8571


Training Epochs:  41%|████▏     | 619/1500 [07:58<11:21,  1.29it/s]

Early stopping at epoch 620
Evaluating on test set with best model...


In [9]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.93      0.87      0.90        30
        African National Congress (South Africa)       1.00      1.00      1.00        30
                                Al-Qaida in Iraq       0.64      0.93      0.76        30
        Al-Qaida in the Arabian Peninsula (AQAP)       0.90      0.87      0.88        30
                                      Al-Shabaab       1.00      0.97      0.98        30
             Basque Fatherland and Freedom (ETA)       0.79      1.00      0.88        30
                                      Boko Haram       0.93      0.87      0.90        30
  Communist Party of India - Maoist (CPI-Maoist)       0.91      0.97      0.94        30
       Corsican National Liberation Front (FLNC)       0.90      0.93      0.92        30
                       Donetsk People's Republic       0.94      1.00      0.97        30
Farabundo

In [10]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [11]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true